This notebooks includes all experiments included in Section DSNU and natural images of the article. 

This code provides the following results:
1. DSNU and natural images in JPG. 
1. DSNU an natural images in raw.

## DSNU and natural images JPG

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu
import matplotlib.pyplot as plt
import argparse

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

from scipy.stats import pearsonr

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape[:2]
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))  # ['A1', 'A2', ...]
jpeg_ext = 'jpg'

crop_size = (2823, 4240)

flat_images, flat_devices = [], []
natural_images, natural_devices = [], []

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'dark', 'jpg')
    natural_dir = os.path.join(base_dir, tag, 'natural', 'jpg')
    
    # Flats - they are the dark ones now
    for img_path in glob(os.path.join(flat_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        flat_images.append(img_path)
        flat_devices.append(device)
        
    # Naturals
    for img_path in glob(os.path.join(natural_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        natural_images.append(img_path)
        natural_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)
natural_images = np.array(natural_images)
natural_devices = np.array(natural_devices)

In [ ]:
print('Computing fingerprints from DARK frames')

fingerprint_devices = sorted(tags)

k = []
for tag in fingerprint_devices:
    print(tag)
    
    dark_dir = os.path.join(base_dir, tag, 'dark', 'jpg')
    
    dark_imgs = []
    for img_path in glob(os.path.join(dark_dir, f'*.{jpeg_ext}')):
        im = np.asarray(Image.open(img_path))
        
        if im.dtype != np.uint8 or im.ndim != 3:
            print('Skipping image:', img_path)
            continue
        
        im_cut = cut_fixed(im, crop_size)
        if im_cut is None:
            print('Skipping (too small):', img_path)
            continue
        
        dark_imgs.append(im_cut.astype(np.float32))
    
    if len(dark_imgs) == 0:
        raise ValueError(f"No valid dark frames for {tag}")
    
    avg_dark = np.mean(dark_imgs, axis=0)
    
    avg_dark_grey = prnu.rgb2gray(avg_dark)
    
    k.append(avg_dark_grey)

k = np.stack(k, 0)

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
print('Computing residuals + matching (streaming)')

matching_cases = []
mismatching_cases = []

for n_idx, img_path in enumerate(natural_images):
    
    im = np.asarray(Image.open(img_path))
    
    im_cut = cut_fixed(im, crop_size)
    if im_cut is None:
        print('Skipping (too small):', img_path)
        continue
    
    natural_w = prnu.extract_single(im_cut)
    
    device_nat = natural_devices[n_idx]

    for f_idx, fingerprint_k in enumerate(k):
        device_fp = fingerprint_devices[f_idx]

        cc2d = prnu.crosscorr_2d(fingerprint_k, natural_w)
        pce_value = prnu.pce(cc2d)['pce']
        
        pearson = pearson_corr(fingerprint_k, natural_w)

        case = {
            'pce': pce_value,
            'pearson': pearson,
            'fingerprint_device': device_fp,
            'natural_device': device_nat,
            'natural_path': img_path
        }

        if device_fp == device_nat:
            matching_cases.append(case)
        else:
            mismatching_cases.append(case)

In [ ]:
np.savez(
    'natural-dsnu-jpg.npz',
    matching_cases=np.array(matching_cases, dtype=object),
    mismatching_cases=np.array(mismatching_cases, dtype=object)
)

In [ ]:
data = np.load("natural-dsnu-jpg.npz", allow_pickle=True)
matching_cases = data["matching_cases"].tolist()
mismatching_cases = data["mismatching_cases"].tolist()

from collections import defaultdict

data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pearson'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pearson'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]

import matplotlib.pyplot as plt
plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)


plt.xticks(range(len(sensors)), sensors, rotation=45)

plt.xlabel("Sensor", fontsize =16)
plt.ylabel("Pearson correlation value", fontsize = 16)
plt.title("Pearson correlation distribution per sensor DSNU with natural images", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)
)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

In [ ]:
from collections import defaultdict

data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pce'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pce'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]

import matplotlib.pyplot as plt
plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)

plt.xticks(range(len(sensors)), sensors, rotation=45)

plt.xlabel("Sensor", fontsize =16)
plt.ylabel("sPCE value", fontsize = 16)
plt.title("sPCE distribution per sensor DSNU with natural images", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

## DSNU and natural images raw

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu_raw
import matplotlib.pyplot as plt
import argparse

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

In [ ]:
def split_bayer(im):
    R  = im[0::2, 0::2]
    G1 = im[0::2, 1::2]
    G2 = im[1::2, 0::2]
    B  = im[1::2, 1::2]
    return [R, G1, G2, B]

In [ ]:
def cut_fixed(img, shape):
    h, w = img.shape[:2]
    ch, cw = shape
    
    if h < ch or w < cw:
        return None
    
    return img[:ch, :cw]


def enforce_even(img):
    H, W = img.shape
    return img[:H - (H % 2), :W - (W % 2)]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))

flat_images, flat_devices = [], []
natural_images, natural_devices = [], []

raw_exts = ['cr3', 'cr2', 'arw']

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'dark', 'raw')
    natural_dir = os.path.join(base_dir, tag, 'natural', 'raw')
    
    # Flats
    for ext in raw_exts:
        for img_path in glob(os.path.join(flat_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            flat_images.append(img_path)
            flat_devices.append(device)
        
    # Naturals
    for ext in raw_exts:
        for img_path in glob(os.path.join(natural_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            natural_images.append(img_path)
            natural_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)
natural_images = np.array(natural_images)
natural_devices = np.array(natural_devices)

In [ ]:
def open_image(img_path):
    img = rawpy.imread(img_path)
    return img.raw_image_visible

In [ ]:
print("Computing global crop size...")

min_H, min_W = np.inf, np.inf
all_images = np.concatenate([flat_images, natural_images])

for img_path in all_images:
    im = np.asarray(open_image(img_path))
    
    im = enforce_even(im)
    H, W = im.shape
    
    min_H = min(min_H, H)
    min_W = min(min_W, W)

crop_size = (int(min_H), int(min_W))
print("Global crop size:", crop_size)

In [ ]:
crop_size = (2848, 4256)
print('Computing DSNU (channel averages)')

fingerprint_devices = sorted(np.unique(flat_devices))
k = []

for device in fingerprint_devices:
    print("Device:", device)
    
    channel_imgs = [[], [], [], []]  # R, G1, G2, B
    
    for img_path in flat_images[flat_devices == device]:
        im = np.asarray(open_image(img_path)).astype(np.float32)
        
        im = enforce_even(im)
        im_crop = cut_fixed(im, crop_size)
        
        if im_crop is None:
            raise ValueError(f"Crop failed in {img_path}")
        
        channels = split_bayer(im_crop)
        
        for c in range(4):
            channel_imgs[c].append(channels[c])
    
    k_device = []
    for c in range(4):
        imgs_c = np.stack(channel_imgs[c], axis=0)
        k_c = np.mean(imgs_c, axis=0)
        k_device.append(k_c)
    
    k.append(k_device)

k = np.array(k)
print("k shape:", k.shape)

In [ ]:
from scipy.stats import pearsonr
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
output_file = 'spce-pearson-raw-nat-dsnu.npz'

if os.path.exists(output_file):
    data = np.load(output_file, allow_pickle=True)
    results = list(data['results'])
    
    processed_imgs = {r['img_idx'] for r in results}
    start_idx = max(processed_imgs) + 1 if processed_imgs else 0
    
    print(f"Restarting from image {start_idx}")
else:
    results = []
    start_idx = 0

print('Processing natural images (streaming)')

for n_idx in range(start_idx, len(natural_images)):
    img_path = natural_images[n_idx]
    device_nat = natural_devices[n_idx]
    
    print(f"Image {n_idx}")

    im = np.asarray(open_image(img_path)).astype(np.float32)
    
    im = enforce_even(im)
    im_crop = cut_fixed(im, crop_size)
    
    if im_crop is None:
        raise ValueError(f"Crop failed in {img_path}")
    
    channels = split_bayer(im_crop)

    w_channels = []
    for c in range(4):
        w_c = prnu_raw.extract_single(channels[c])
        w_channels.append(w_c)

    for f_idx, fingerprint_k in enumerate(k):
        device_fp = fingerprint_devices[f_idx]
        
        for c in range(4):
            cc2d = prnu_raw.crosscorr_2d(fingerprint_k[c], w_channels[c])
            pce_value = prnu_raw.pce(cc2d)['pce']
            
            pearson = pearson_corr(fingerprint_k[c], w_channels[c])
            
            results.append({
                'pce': pce_value,
                'pearson': pearson,
                'match': int(device_fp == device_nat),
                'fp_device': device_fp,
                'nat_device': device_nat,
                'img_idx': n_idx,
                'channel': c
            })

    if n_idx % 5 == 0:
        print("Saving checkpoint...")
        np.savez(output_file, results=results)

np.savez(output_file, results=results)

In [ ]:
from collections import defaultdict
import os

channel_map = {
    0: 'B',
    1: 'G1',
    2: 'G2',
    3: 'R'
}

output_dir = "raw-violins-dsnu-nat"
os.makedirs(output_dir, exist_ok=True)

data = np.load('spce-pearson-raw-nat-dsnu.npz', allow_pickle=True)
results = data['results']

channel_names = sorted(list(set(channel_map[case['channel']] for case in results)))

for cname in channel_names:

    sensor_data_pce = defaultdict(lambda: {'match': [], 'mismatch': []})
    sensor_data_pearson = defaultdict(lambda: {'match': [], 'mismatch': []})

    for case in results:
        if channel_map[case['channel']] != cname:
            continue

        sensor = case['fp_device']
        
        if case['match']:
            sensor_data_pce[sensor]['match'].append(case['pce'])
            sensor_data_pearson[sensor]['match'].append(case['pearson'])
        else:
            sensor_data_pce[sensor]['mismatch'].append(case['pce'])
            sensor_data_pearson[sensor]['mismatch'].append(case['pearson'])

    sensors = sorted(sensor_data_pce.keys())

    positions_match = [i - 0.15 for i in range(len(sensors))]
    positions_mismatch = [i + 0.15 for i in range(len(sensors))]

    match_values = [sensor_data_pce[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pce[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.yscale('log')
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("sPCE value", fontsize=16)
    plt.title(f"sPCE distribution per sensor - Channel {cname} DSNU vs Natural", fontsize=16)

    plt.grid(True, which="both", linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pce_{cname}.png"), dpi=300)
    plt.close()

    match_values = [sensor_data_pearson[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pearson[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("Pearson correlation", fontsize=16)
    plt.title(f"Pearson distribution per sensor - Channel {cname} DSNU vs Natural", fontsize=16)

    plt.grid(True, linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pearson_{cname}.png"), dpi=300)
    plt.close()